In [13]:
import pandas as pd
import json

In [14]:
file_path = '/Users/juanpazmino/Documents/Desarrollo/Personal_projects/data_visualization/Lab01/01mdi_homicidios_intencionales_pm_2014_2025.xlsx'
df = pd.read_excel(file_path, sheet_name='1', header=1)
df = df.drop(df.columns[0], axis=1)
df.head()

,tipo_muerte,zona,subzona,distrito,circuito,codigo_subcircuito,subcircuito,codigo_provincia,provincia,codigo_canton,...,medida_edad,sexo,genero,etnia,estado_civil,nacionalidad,discapacidad,profesion_registro_civil,instruccion,antecedentes
0,ASESINATO,ZONA 1,ESMERALDAS,ESMERALDAS,LAS PALMAS,08D01C02S01,LAS PALMAS 1,8,ESMERALDAS,801,...,A,MUJER,FEMENINO,AFRO,SOLTERO,ECUADOR,NINGUNA,ESTADO PERSONAL,SIN_DATO,SIN_DATO
1,ASESINATO,ZONA 4,MANABÍ,PORTOVIEJO,SAN PABLO,13D01C05S02,SAN PABLO 2,13,MANABÍ,1301,...,A,HOMBRE,MASCULINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,TRABAJADOR GENERAL,BASICA,SIN_DATO
2,ASESINATO,ZONA 4,MANABÍ,PORTOVIEJO,SAN PABLO,13D01C05S02,SAN PABLO 2,13,MANABÍ,1301,...,A,HOMBRE,MASCULINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,MAESTRO DE OBRA/CONSTRUCCIÓN,SIN_DATO,SIN_DATO
3,ASESINATO,ZONA 4,MANABÍ,MANTA,LA PILA,13D02C14S01,LA PILA 1,13,MANABÍ,1309,...,A,MUJER,FEMENINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,SIN_DATO,SIN_DATO,SIN_DATO
4,ASESINATO,ZONA 4,MANABÍ,MANTA,LA PILA,13D02C14S01,LA PILA 1,13,MANABÍ,1309,...,A,MUJER,FEMENINO,MESTIZO/A,SOLTERO,ECUADOR,NINGUNA,ESTADO PERSONAL,SECUNDARIA,SIN_DATO


In [ ]:
import pandas as pd
import json
import unicodedata

# Paleta de colores para los polígonos del Streamgraph
COLORES = ["#64af59", "#ff9b4e", "#a081b9", "#ffff7c", "#6987a9", 
           "#d95749", "#007300", "#c44500", "#6831a2", "#c4cc00", 
           "#004fa1", "#ad0000", "#338f33", "#c85110", "#b498d1"]

# 1. Función extrema para limpiar textos (sin tildes, todo mayúscula)
def limpiar_texto(texto):
    texto = str(texto).strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ASCII', 'ignore').decode('utf-8')
    return texto.upper()

def generar_archivos_mensuales(df):
    print("Iniciando procesamiento directo para MultiStream...")
    
    df_opt = df[['fecha_infraccion', 'provincia', 'canton']].copy()
    df_opt = df_opt.replace(["N/A", "n/a", "N/D", ""], "DESCONOCIDO").fillna("DESCONOCIDO")

    # 2. Limpieza y desambiguación 
    df_opt['provincia'] = df_opt['provincia'].apply(limpiar_texto)
    df_opt['canton'] = df_opt['canton'].apply(limpiar_texto)
    df_opt['provincia'] = df_opt['provincia'] + " (PROV)"

    # 3. GRANULARIDAD MENSUAL (Evita picos matemáticos extremos)
    df_opt['fecha_infraccion'] = pd.to_datetime(df_opt['fecha_infraccion'], errors='coerce')
    df_opt = df_opt.dropna(subset=['fecha_infraccion'])
    df_opt = df_opt[(df_opt['fecha_infraccion'].dt.year >= 2014) & (df_opt['fecha_infraccion'].dt.year <= 2024)]
    
    # Formato ISO-8601 estricto requerido por MultiStream
    df_opt['date'] = df_opt['fecha_infraccion'].dt.strftime('%Y-%m-01T00:00:00.000Z')
    fechas_todas = pd.date_range(start='2014-01-01', end='2024-12-01', freq='MS').strftime('%Y-%m-01T00:00:00.000Z').tolist()

    # 4. Extraer Jerarquía (Top 5 Provincias, Top 3 Cantones)
    top_provs = df_opt['provincia'].value_counts().nlargest(5).index.tolist()
    df_top = df_opt[df_opt['provincia'].isin(top_provs)].copy()

    mapa_prov_cantones = {}
    cantones_validos = []
    
    for prov in top_provs:
        top_c = df_top[df_top['provincia'] == prov]['canton'].value_counts().nlargest(3).index.tolist()
        
        # PARCHE: MultiStream colapsa si una rama tiene un solo hijo. Agregamos uno fantasma.
        if len(top_c) == 1:
            top_c.append(f"OTROS EN {prov}")
            
        mapa_prov_cantones[prov] = top_c
        cantones_validos.extend(top_c)
        
    df_final = df_top[df_top['canton'].isin(cantones_validos)].copy()

    # 5. Agregaciones de Datos (Hojas, Nodos y Raíz)
    hojas_pivot = df_final.groupby(['date', 'canton']).size().reset_index(name='count').pivot(index='date', columns='canton', values='count').reindex(fechas_todas).fillna(0).astype(int)
    prov_pivot = df_final.groupby(['date', 'provincia']).size().reset_index(name='count').pivot(index='date', columns='provincia', values='count').reindex(fechas_todas).fillna(0).astype(int)
    pais_pivot = df_final.groupby(['date']).size().reindex(fechas_todas).fillna(0).astype(int)

    # =========================================================================
    # 6. CONSTRUCCIÓN DIRECTA DEL JSON RENDERIZADO (Reemplaza a Node.js)
    # =========================================================================
    root_name = "ECUADOR"
    root_key = "R0"
    
    ranges = {
        "name": root_name,
        "depth": 0,
        "x": 0.5,
        "y": 0.0,
        "key": root_key,
        "color": "#a65628",
        "children": []
    }
    
    data_records = []
    total_hojas = sum(len(c) for c in mapa_prov_cantones.values())
    hoja_actual = 0
    c_idx = 0
    
    # Armar la estructura del árbol y coordenadas
    for i, prov in enumerate(top_provs):
        prov_key = f"{root_key}_{i}"
        hijos = mapa_prov_cantones[prov]
        
        x_prov_start = hoja_actual / total_hojas
        x_prov_end = (hoja_actual + len(hijos)) / total_hojas
        x_prov = (x_prov_start + x_prov_end) / 2.0
        
        prov_node = {
            "name": prov,
            "depth": 1,
            "x": x_prov,
            "y": 0.5,
            "key": prov_key,
            "color": COLORES[c_idx % len(COLORES)],
            "children": []
        }
        c_idx += 1
        
        for j, canton in enumerate(hijos):
            canton_key = f"{prov_key}_{j}"
            x_canton = (hoja_actual + 0.5) / total_hojas
            hoja_actual += 1
            
            canton_node = {
                "name": canton,
                "depth": 2,
                "x": x_canton,
                "y": 1.0,
                "key": canton_key,
                "color": COLORES[c_idx % len(COLORES)],
                "visible": True,
                "img": ""
            }
            c_idx += 1
            prov_node["children"].append(canton_node)
            
        ranges["children"].append(prov_node)

    # Armar la estructura del array de datos
    for date in fechas_todas:
        # Raíz
        val_root = int(pais_pivot.at[date]) if date in pais_pivot.index else 0
        data_records.append({"date_time": date, "category": root_name, "value": val_root, "key": root_key})
        
        for i, prov in enumerate(top_provs):
            prov_key = f"{root_key}_{i}"
            val_prov = int(prov_pivot.at[date, prov]) if prov in prov_pivot.columns else 0
            data_records.append({"date_time": date, "category": prov, "value": val_prov, "key": prov_key})
            
            for j, canton in enumerate(mapa_prov_cantones[prov]):
                canton_key = f"{prov_key}_{j}"
                val_canton = int(hojas_pivot.at[date, canton]) if canton in hojas_pivot.columns else 0
                data_records.append({"date_time": date, "category": canton, "value": val_canton, "key": canton_key})

    # Generar JSON estructurado
    output_json = {
        "ranges": ranges,
        "data": data_records,
        "type": "cantidad de homicidios",
        "t_granularity": "months",
        "t_step": 1
    }
    
    with open('multistream_listo_para_pegar.json', 'w', encoding='utf-8') as f:
        json.dump(output_json, f, ensure_ascii=False, indent=4)
        
    print("¡Proceso 100% exitoso!")
    print("Ya no necesito Node.js. El archivo 'multistream_listo.json' es el que funciona.")

# Solo ejecuta enviando tu dataframe:
generar_archivos_mensuales(df)

Iniciando procesamiento directo para MultiStream...
¡Proceso 100% exitoso!
Ya no necesitas Node.js. Sube el archivo 'multistream_listo_para_pegar.json' directamente a la web.
